In [1]:
!pip install  pdfplumber faiss-cpu sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 86.4 MB/s eta 0:00:00


In [2]:
import os
folder_path = "/content"
for file in os.listdir(folder_path):
  if file.endswith(".pdf"):
    os.remove(os.path.join(folder_path, file))
print("Old files removed")

Old files removed


In [3]:
from google.colab import files
uploaded = files.upload()

Saving software_engineer.pdf to software_engineer.pdf
Saving Data_Scientist_Resume.pdf to Data_Scientist_Resume.pdf
Saving DevOps_Engineer_Resume.pdf to DevOps_Engineer_Resume.pdf
Saving Cybersecurity_Analyst_Resume.pdf to Cybersecurity_Analyst_Resume.pdf
Saving Software_Engineer_Resume.pdf to Software_Engineer_Resume.pdf


In [4]:
import pdfplumber

def extract_text_from_pdf(file):
    text = ""
    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            content = page.extract_text()
            if content:
                text += content + "\n"
    return text

In [5]:
import re

def extract_sections(text):
    sections = {
        "skills": "",
        "experience": "",
        "projects": "",
        "certifications": ""
    }

    text_lower = text.lower()


    patterns = {
        "skills": ["skills", "technical skills", "key skills"],
        "experience": ["experience", "work experience", "professional experience"],
        "projects": ["projects", "personal projects"],
        "certifications": ["certifications", "certificates"]
    }

    for key, keywords in patterns.items():
        for kw in keywords:
            pattern = rf"{kw}(.+?)(\n[A-Z ]{{3,}}|\Z)"
            match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
            if match:
                sections[key] = match.group(1).strip()
                break

    return sections

In [6]:
resume_texts = {}

for file_name in uploaded.keys():
    text = extract_text_from_pdf(file_name)
    resume_texts[file_name] = text

In [7]:
def clean_text(text):
    text = text.lower()
    text = text.replace("\n", " ")
    text = " ".join(text.split())
    return text[:2000]

In [8]:
structured_data = {}

for file_name, text in resume_texts.items():
    structured_data[file_name] = extract_sections(text)

In [9]:
cleaned_docs = []

for file_name, sections in structured_data.items():
    combined_text = clean_text(" ".join(sections.values()))

    cleaned_docs.append({
        "file": file_name,
        "text": combined_text
    })

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2', device ='cpu')

In [11]:
import numpy as np

documents = []
embeddings = []

texts = [doc["text"] for doc in cleaned_docs]
documents = [doc["file"] for doc in cleaned_docs]

embeddings = embed_model.encode(
    texts,
    batch_size=4
)

In [12]:
import numpy as np
import faiss

embeddings = np.array(embeddings).astype('float32')

faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

In [13]:
query = input("Enter Job Description: ")

query_embedding = embed_model.encode([query]).astype('float32')

faiss.normalize_L2(query_embedding)

D, I = index.search(query_embedding, k=min(5, len(documents)))

print("\n Top Matching resumes:\n")

for rank, (idx, dist) in enumerate(zip(I[0], D[0]), start=1):
    percentage = round(dist * 100, 2)

    print(f"Rank {rank}: {documents[idx]}")
    print(f"Match Score: {percentage}%")
    print("-" * 40)

Enter Job Description: Company Name] is looking for an experienced DevOps engineer with strong technical expertise in CI/CD pipelines, infrastructure automation, and cloud platforms, along with excellent collaboration and communication skills. The candidate should have hands-on experience with configuration management tools, a solid understanding of DevOps practices, and a working knowledge of internal backend systems. The ideal candidate will have the ability to coordinate and bridge gaps between the software developer and the operation team, ensuring a smooth workflow.

 Top Matching resumes:

Rank 1: DevOps_Engineer_Resume.pdf
Match Score: 58.13999938964844%
----------------------------------------
Rank 2: Software_Engineer_Resume.pdf
Match Score: 36.95000076293945%
----------------------------------------
Rank 3: Cybersecurity_Analyst_Resume.pdf
Match Score: 35.20000076293945%
----------------------------------------
Rank 4: software_engineer.pdf
Match Score: 34.45000076293945%
---